# PARC2026 06: GPU / LIBERO Validation

Google Driveを正本として、実Checkpoint＋実RLDSのGPU推論を検証します。任意で、公式LIBERO環境に予測Actionを入力する因果的な閉ループ1 Trialも実行します。

このNotebookは学習を行いません。公式Benchmarkの初期状態や観測は評価目的だけに使い、学習データへ追加しません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/yu37330/Physical_ai.git'
BRANCH = 'agent/add-colab-agent-cockpit'
!rm -rf /content/Physical_ai
!git clone --branch $BRANCH $REPO_URL /content/Physical_ai
%cd /content/Physical_ai
!PROJECT_ROOT=/content/Physical_ai WORKDIR=/content/openvla-oft bash training/openvla_oft_a100/scripts/bootstrap_colab.sh
!OPENVLA_OFT_SOURCE=/content/openvla-oft bash submission/openvla_oft_offline/scripts/prepare_vendor.sh
!pip install -q -r training/openvla_oft_a100/requirements-data.txt
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 1. Drive上の実データを指定

`RLDS_DATASET_DIR`には`dataset_info.json`が存在するTFDS Builder directoryを指定します。CheckpointはDriveからColabローカルへ差分コピーして使用します。

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/PARC2026')
os.environ['PHYSICAL_AI_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

CHECKPOINT_DRIVE_DIR = DRIVE_ROOT / '30_models/openvla_oft_plus'
CHECKPOINT_WORK_DIR = Path('/content/work/openvla_oft_plus')
RLDS_DATASET_DIR = DRIVE_ROOT / '20_processed/rlds/parc_libero_plus_selected/1.0.0'
GPU_REPORT = DRIVE_ROOT / '40_experiments/gpu_validation/latest_gpu_validation.json'

assert CHECKPOINT_DRIVE_DIR.is_dir(), CHECKPOINT_DRIVE_DIR
assert RLDS_DATASET_DIR.is_dir(), RLDS_DATASET_DIR
CHECKPOINT_WORK_DIR.mkdir(parents=True, exist_ok=True)
print('Checkpoint:', CHECKPOINT_DRIVE_DIR)
print('RLDS:', RLDS_DATASET_DIR)
print('Report:', GPU_REPORT)

In [ ]:
# Google Driveは正本、/contentは実行時キャッシュ
!rsync -ah --delete --info=progress2 "{CHECKPOINT_DRIVE_DIR}/" "{CHECKPOINT_WORK_DIR}/"

## 2. 実Checkpoint＋実RLDS GPU Gate

Warm-up後に複数Frameを推論し、CUDA環境、Action chunk shape、有限値、Latency、Peak VRAM、教師Actionとの誤差をJSONへ保存します。

In [ ]:
!python scripts/run_colab_gpu_validation.py \
  --checkpoint-dir "{CHECKPOINT_WORK_DIR}" \
  --dataset-dir "{RLDS_DATASET_DIR}" \
  --split val \
  --episode-offset 0 \
  --start-frame 0 \
  --num-frames 3 \
  --warmup-runs 1 \
  --output "{GPU_REPORT}"

In [ ]:
import json
gpu_report = json.loads(GPU_REPORT.read_text(encoding='utf-8'))
assert gpu_report['status'] == 'pass', gpu_report
gpu_report['summary']

## 3. 任意: LIBERO因果閉ループ1 Trial

以下は記録済みReplayではありません。PolicyのActionを`OffScreenRenderEnv`へ入力し、得られた次観測で再推論します。まずTask 0／初期状態0の1 Trialだけで接続確認します。公式評価の画像保存は標準で無効です。

In [ ]:
RUN_LIBERO_CLOSED_LOOP = False  # GPU Gate通過後にTrueへ変更

if RUN_LIBERO_CLOSED_LOOP:
    !LIBERO_WORKDIR=/content/LIBERO OPENVLA_OFT_WORKDIR=/content/openvla-oft bash training/openvla_oft_a100/scripts/bootstrap_libero_colab.sh

In [ ]:
from datetime import datetime

if RUN_LIBERO_CLOSED_LOOP:
    RUN_ID = datetime.now().strftime('libero_spatial_t0_%Y%m%d_%H%M%S')
    !python scripts/run_libero_closed_loop.py \
      --checkpoint-dir "{CHECKPOINT_WORK_DIR}" \
      --task-suite-name libero_spatial \
      --task-id 0 \
      --init-state-id 0 \
      --seed 7 \
      --max-steps 300 \
      --run-id "{RUN_ID}"

## 判定順

1. GPUレポートが`pass`
2. Peak VRAMとLatencyが許容範囲
3. Action chunkが`(8, 7)`かつ有限値
4. LIBERO 1 Trialが例外なく終了
5. その後にTask数・初期状態数を増やして成功率を評価

閉ループ成功率を学習へ自動フィードバックする処理は、このNotebookには含めません。